# Notes

# Import

In [1]:
from dotenv import load_dotenv
import os 
import fatsecret

# Исполнение

In [6]:
load_dotenv()

Client_ID = os.getenv('ConsumerKey')
Client_Secret = os.getenv('ConsumerSecret')

consumer_key, secret_key = Client_ID, Client_Secret
CONSUMER_KEY, CONSUMER_SECRET = Client_ID, Client_Secret
CLIENT_ID, CLIENT_SECRET = Client_ID, Client_Secret

In [8]:
import os
from dotenv import load_dotenv
from requests_oauthlib import OAuth1Session

# Ключевой момент: signature_type='BODY' заставляет передавать OAuth-параметры
# в теле POST-запроса (application/x-www-form-urlencoded), а не в заголовке
session = OAuth1Session(
    client_key=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    callback_uri='oob',
    signature_type='BODY'          # <-- решает проблему
)

url = 'https://authentication.fatsecret.com/oauth/request_token'
resp = session.post(url)

print(f'Status: {resp.status_code}')
print(f'Response: {resp.text}')

if resp.status_code == 200:
    from urllib.parse import parse_qs
    data = parse_qs(resp.text)
    request_token = data.get('oauth_token', [None])[0]
    request_token_secret = data.get('oauth_token_secret', [None])[0]
    print(f'\n✅ request_token = {request_token}')
    print(f'   request_token_secret = {request_token_secret}')
else:
    print('\n❌ Ошибка: проверьте Client_ID и Client_Secret в .env')

Status: 200
Response: oauth_callback_confirmed=true&oauth_token=50495912789a48a3ae8c335a7921e95b&oauth_token_secret=97fc75decfc0489eab9805731f4ec58e

✅ request_token = 50495912789a48a3ae8c335a7921e95b
   request_token_secret = 97fc75decfc0489eab9805731f4ec58e


In [ ]:
import webbrowser
from urllib.parse import parse_qs
from requests_oauthlib import OAuth1Session


# --- Шаг 2: авторизация пользователя ---
auth_url = f'https://authentication.fatsecret.com/oauth/authorize?oauth_token={request_token}'
print(f"Откройте ссылку в браузере и авторизуйте приложение:\n{auth_url}")
webbrowser.open(auth_url)

# Пользователь вводит PIN (код подтверждения) после успешной авторизации
verifier = input("\nВведите PIN-код, который вы получили: ").strip()

# --- Шаг 3: обмен request_token на access_token ---
access_session = OAuth1Session(
    CLIENT_ID,
    client_secret=CLIENT_SECRET,
    resource_owner_key=request_token,
    resource_owner_secret=request_token_secret,
    verifier=verifier,
    signature_type='BODY'          # обязательно для FatSecret
)

resp = access_session.post('https://authentication.fatsecret.com/oauth/access_token')
print(f"Ответ сервера: {resp.status_code} {resp.text}")

if resp.status_code == 200:
    data = parse_qs(resp.text)
    access_token = data.get('oauth_token', [None])[0]
    access_token_secret = data.get('oauth_token_secret', [None])[0]
    print(f"\n✅ Access token получен:\n  oauth_token = {access_token}\n  oauth_token_secret = {access_token_secret}")
else:
    print("❌ Ошибка получения access token")
    exit()

# --- Шаг 4: пример API-вызова (получение профиля пользователя) ---
api_session = OAuth1Session(
    CLIENT_ID,
    client_secret=CLIENT_SECRET,
    resource_owner_key=access_token,
    resource_owner_secret=access_token_secret,
    signature_type='BODY'
)

api_url = 'https://platform.fatsecret.com/rest/server.api'
params = {
    'method': 'profile.get',
    'format': 'json'
}

response = api_session.get(api_url, params=params)
print(f"\nAPI ответ (profile.get): {response.status_code}")
if response.status_code == 200:
    print(response.json())
else:
    print(response.text)

Откройте ссылку в браузере и авторизуйте приложение:
https://authentication.fatsecret.com/oauth/authorize?oauth_token=50495912789a48a3ae8c335a7921e95b
